# BigMart v67–v69 audit

## tl;dr

v67 has a **calculated**, not newly observed, public RMSE of 1124.1799. v68 and v69 are unscored trained refinements. The v69 recipe improved two fresh-model CV checks by 4.5516 and 3.3099 RMSE points; this is not a public-score forecast.

## Context & Methods

The submission generators `make_submission.py` and `refinement.py` are the training source. The latter retrains old and suffix-only corrections from raw data on 15 folds. `validate.py` rebuilds a fresh proxy baseline inside outer folds.

### Key Assumptions

Reported scores match the hashed CSVs; public rows remain fixed. The historical ensemble is a frozen input. Repeated CV uses the same training dataset and is subject to selection bias. Small exact-value tables are used here because this notebook verifies calculations and artifacts, rather than presenting a trend or a graphical model comparison.

In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
root = Path.cwd()
data = root.parent / 'data'
if not data.exists():
    data = Path('/Users/abhimanyunain/Downloads')
manifest = json.loads((root / 'v67_manifest.json').read_text())
refinement = json.loads((root / 'refinement_manifest.json').read_text())

## Data

Raw inputs and frozen scored files are identified by SHA-256, not just filenames.

In [2]:
records = []
for name, item in refinement['inputs'].items():
    path = data / Path(item['path']).name
    observed = hashlib.sha256(path.read_bytes()).hexdigest()
    assert observed == item['sha256']
    records.append({'input': name, 'hash_verified': observed == item['sha256']})
print(pd.DataFrame(records).to_string(index=False))

   input  hash_verified
   train           True
    test           True
  sample           True
baseline           True


## Results

### Exact affine-line score calculation

In [3]:
x = np.array([0., .75, 1.])
s = np.array([1140.0228905024, 1126.0148395277, 1124.4058880453])
a,b,c = np.linalg.solve(np.c_[x*x,x,np.ones(3)],s*s)
alpha = -b/(2*a)
calculated = np.sqrt(c-b*b/(4*a))
assert abs(calculated-manifest['calculated_public_rmse']) < 1e-8
print(f'Optimal alpha: {alpha:.12f}\nCalculated public RMSE: {calculated:.10f}\nObserved v67 public RMSE: not supplied')

Optimal alpha: 1.135112248150
Calculated public RMSE: 1124.1798633738
Observed v67 public RMSE: not supplied


### Fresh-model validation

These are proxy training-data checks, not leaderboard results.

In [4]:
rows=[]
for filename in ['validation_11827.json','validation_8461_grouped.json']:
    check=json.loads((root/filename).read_text())
    rows.append({'check': 'Product holdout' if check['grouped_by_item'] else 'Outlet stratified',
                 'baseline': check['baseline_rmse'],
                 'v68_recipe': check['variants']['1.0']['rmse'],
                 'v69_recipe': check['variants']['1.5']['rmse']})
print(pd.DataFrame(rows).round(4).to_string(index=False))

            check  baseline  v68_recipe  v69_recipe
Outlet stratified 1066.5368   1062.7165   1061.9852
  Product holdout 1065.3137   1062.2886   1062.0038


### Submission integrity

In [5]:
keys=['Item_Identifier','Outlet_Identifier']; target='Item_Outlet_Sales'
test=pd.read_csv(data/'test_AbJTz2l.csv')
base=pd.read_csv(data/'bigmart_submission_v67_exact_quantity_optimum.csv')
outputs={}
for version,item in refinement['output'].items():
    path=data/Path(item['path']).name
    assert hashlib.sha256(path.read_bytes()).hexdigest()==item['sha256']
    frame=pd.read_csv(path)
    assert frame.shape==(5681,3) and frame[keys].equals(test[keys])
    assert np.isfinite(frame[target]).all() and (frame[target]>=0).all()
    outputs[version]=frame[target].to_numpy()
assert np.allclose(outputs['v69']-base[target],1.5*(outputs['v68']-base[target]),atol=1e-8,rtol=0)
print('Passed: row count, column order, identifiers, hashes, finite/nonnegative values and affine relation.')

Passed: row count, column order, identifiers, hashes, finite/nonnegative values and affine relation.


## Takeaways

Try v69 first and v68 second. AV evaluation is required to establish whether either reaches 1120. No hidden labels or inferred public-row membership were used. See `README.md` for exact generator commands and baseline reproduction limits.